# INV-M365-DO - Standalone Graph Runtime Pack 0.1.2 Readiness Fix Invariants v1

**Plan reference:** `plan:m365-standalone-graph-runtime-pack-0-1-2-readiness-fix:C2`
**Mission:** Establish notebook-backed evidence for the five readiness-fix invariants that back every C1-C8 code/package edit in the `0.1.2` corrective pack release.
**Mode:** deterministic-static (no random sampling, no live network).
**Lemma bundle:** `L99_m365_standalone_graph_runtime_pack_0_1_2_readiness_fix_v1`

## Sections
- C2.0 - Setup, seed lock, target truth
- C2.1 - V_CORRECT: version-correction invariant
- C2.2 - A_SELFDESC: artifact-layout / installed-payload readiness invariant
- C2.3 - D_CLOSED: dependency-contract invariant
- C2.4 - S_REAL: real local-socket UCP-client acceptance invariant
- C2.5 - P_REPRO: provenance-reproducibility invariant
- C2.6 - Bundled predicate `PackReady_0_1_2`
- C2.7 - Negative cases (each clause broken in isolation)
- C2.8 - Emit deterministic verification JSON


In [ ]:
# C2.0 - Setup, seed lock, target truth
import json, os, random
from pathlib import Path

PYTHONHASHSEED = 0
PYTHON_RANDOM_SEED = 0
NUMPY_SEED = 0
os.environ['PYTHONHASHSEED'] = str(PYTHONHASHSEED)
random.seed(PYTHON_RANDOM_SEED)

TARGET = {
    'pack_id': 'com.smarthaus.m365',
    'pack_version': '0.1.2',
    'runtime_version': '0.1.2',
    'bundle_name': 'com.smarthaus.m365-0.1.2.ucp.tar.gz',
    'install_dir': '/Users/smarthaus/Projects/GitHub/IntegrationPacks/M365/0.1.2/',
    'historical_versions': ['1.1.0', '1.1.1'],
}
TARGET


## C2.1 - V_CORRECT: version-correction invariant

**Statement:** `V_CORRECT(p)` is true when, for the active marketplace pack `p`, the pack manifest `version`, the runtime `RUNTIME_VERSION` constant, the runtime `/v1/runtime/version` response, the bundle filename, and the install directory path all read `0.1.2`. Historical `1.1.0` and `1.1.1` references appear only as labeled evidence text and never as the active release authority.

In [ ]:
def V_CORRECT(state):
    return (
        state.get('manifest_version') == '0.1.2'
        and state.get('runtime_version_constant') == '0.1.2'
        and state.get('runtime_version_endpoint') == '0.1.2'
        and state.get('bundle_name') == 'com.smarthaus.m365-0.1.2.ucp.tar.gz'
        and state.get('install_dir', '').rstrip('/').endswith('/IntegrationPacks/M365/0.1.2')
        and not any(state.get('active_release_claims', {}).get(v, False) for v in ('1.1.0', '1.1.1'))
    )

GREEN_V = {
    'manifest_version': '0.1.2',
    'runtime_version_constant': '0.1.2',
    'runtime_version_endpoint': '0.1.2',
    'bundle_name': 'com.smarthaus.m365-0.1.2.ucp.tar.gz',
    'install_dir': '/Users/smarthaus/Projects/GitHub/IntegrationPacks/M365/0.1.2/',
    'active_release_claims': {'0.1.2': True, '1.1.0': False, '1.1.1': False},
}
assert V_CORRECT(GREEN_V) is True
V_CORRECT(GREEN_V)


## C2.2 - A_SELFDESC: artifact-layout / readiness invariant

**Statement:** `A_SELFDESC(L)` is true when the installed-payload layout `L` is self-describing: after extracting the outer `.ucp.tar.gz` and inner `payload.tar.gz`, launching the runtime from the chosen `installed_root` causes `probe_artifact()` to return `True` without consulting the source repo, sibling repos, or `M365_REPO_ROOT`. Equivalently, every file in `REQUIRED_INSTALLED_FILES` exists at `installed_root` after extraction.

In [ ]:
REQUIRED_INSTALLED_FILES_CANDIDATES = {
    'manifest_at_payload_root': ('manifest.json', 'conformance.json', 'provenance.json', 'setup_schema.json', 'registry/agents.yaml'),
    'envelope_resolution': ('manifest.json', 'setup_schema.json', 'registry/agents.yaml'),
}

def A_SELFDESC(layout):
    contract = layout.get('contract')
    if contract not in REQUIRED_INSTALLED_FILES_CANDIDATES:
        return False
    required = REQUIRED_INSTALLED_FILES_CANDIDATES[contract]
    files_at_root = set(layout.get('files_at_installed_root', []))
    if not all(f in files_at_root for f in required):
        return False
    if layout.get('reads_source_repo', False):
        return False
    if layout.get('reads_sibling_repo', False):
        return False
    if layout.get('reads_m365_repo_root_env', False):
        return False
    return True

GREEN_A = {
    'contract': 'manifest_at_payload_root',
    'files_at_installed_root': [
        'manifest.json', 'conformance.json', 'provenance.json',
        'setup_schema.json', 'registry/agents.yaml',
        'm365_runtime/__init__.py', 'ucp_m365_pack/__init__.py',
    ],
    'reads_source_repo': False,
    'reads_sibling_repo': False,
    'reads_m365_repo_root_env': False,
}
assert A_SELFDESC(GREEN_A) is True
A_SELFDESC(GREEN_A)


## C2.3 - D_CLOSED: dependency-contract invariant

**Statement:** `D_CLOSED(D)` is true when the pack declares its runtime Python dependencies in a manifest/lock readable from the installed payload, the launcher boots into a deterministic dependency probe before binding the listener, and any missing module surfaces as a structured `dependency_missing` outcome (with the missing module name list) instead of a raw `ModuleNotFoundError`. Optional certificate/JWT branches must not block delegated `device_code` import unless declared.

In [ ]:
def D_CLOSED(dep_state):
    if not dep_state.get('declared_dependency_manifest_present', False):
        return False
    if not dep_state.get('startup_probe_present', False):
        return False
    if dep_state.get('missing_module_outcome_class') != 'dependency_missing':
        return False
    if dep_state.get('raw_module_not_found_observed', False):
        return False
    if dep_state.get('depends_on_repo_dot_venv', False):
        return False
    if dep_state.get('jwt_blocks_device_code_import', False):
        return False
    return True

GREEN_D = {
    'declared_dependency_manifest_present': True,
    'startup_probe_present': True,
    'missing_module_outcome_class': 'dependency_missing',
    'raw_module_not_found_observed': False,
    'depends_on_repo_dot_venv': False,
    'jwt_blocks_device_code_import': False,
}
assert D_CLOSED(GREEN_D) is True
D_CLOSED(GREEN_D)


## C2.4 - S_REAL: real local-socket UCP-client acceptance invariant

**Statement:** `S_REAL(R)` is true when release acceptance: (i) extracts the outer `.ucp.tar.gz` and inner `payload.tar.gz`, (ii) launches `python -m m365_runtime --host 127.0.0.1 --port <dynamic>` as a subprocess, (iii) sets `M365_RUNTIME_URL`, (iv) calls `ucp_m365_pack.client.execute_m365_action()` against the live local socket without monkeypatching `_http_runtime_invoke`, (v) proves at least one legacy alias mapping over real HTTP, and (vi) fails closed if `_http_runtime_invoke` is patched in release acceptance.

In [ ]:
def S_REAL(accept):
    return (
        accept.get('extracts_outer_bundle', False)
        and accept.get('extracts_inner_payload', False)
        and accept.get('launches_runtime_subprocess', False)
        and accept.get('binds_loopback_dynamic_port', False)
        and accept.get('sets_m365_runtime_url', False)
        and accept.get('calls_execute_m365_action_unpatched', False)
        and accept.get('http_transport_class') == 'real_httpx'
        and accept.get('http_runtime_invoke_patched', False) is False
        and accept.get('legacy_alias_proven_over_http', False)
        and accept.get('guard_fails_when_http_runtime_invoke_patched', False)
    )

GREEN_S = {
    'extracts_outer_bundle': True,
    'extracts_inner_payload': True,
    'launches_runtime_subprocess': True,
    'binds_loopback_dynamic_port': True,
    'sets_m365_runtime_url': True,
    'calls_execute_m365_action_unpatched': True,
    'http_transport_class': 'real_httpx',
    'http_runtime_invoke_patched': False,
    'legacy_alias_proven_over_http': True,
    'guard_fails_when_http_runtime_invoke_patched': True,
}
assert S_REAL(GREEN_S) is True
S_REAL(GREEN_S)


## C2.5 - P_REPRO: provenance-reproducibility invariant

**Statement:** `P_REPRO(P)` is true when two consecutive deterministic builds from the same recorded source commit produce byte-identical bundle SHAs, the recorded provenance includes source commit, branch, clean/dirty state, dependency lock hash, payload SHA, bundle SHA, manifest SHA, and conformance SHA, and the recorded source commit is `clean`. A final `GO` requires `clean == True`; a `dirty` worktree forces explicit dirty-tree digest and remains `NO_GO` unless the CTO accepts the weaker state.

In [ ]:
def P_REPRO(prov):
    if prov.get('build_a_bundle_sha') != prov.get('build_b_bundle_sha'):
        return False
    required_keys = {
        'source_commit', 'source_branch', 'source_clean',
        'dependency_lock_sha', 'payload_sha', 'bundle_sha',
        'manifest_sha', 'conformance_sha',
    }
    if not required_keys.issubset(prov.keys()):
        return False
    if prov.get('claims_clean') and not prov.get('source_clean'):
        return False
    return True

GREEN_P = {
    'build_a_bundle_sha': 'a' * 64,
    'build_b_bundle_sha': 'a' * 64,
    'source_commit': 'a' * 40,
    'source_branch': 'development',
    'source_clean': True,
    'claims_clean': True,
    'dependency_lock_sha': 'b' * 64,
    'payload_sha': 'c' * 64,
    'bundle_sha': 'a' * 64,
    'manifest_sha': 'd' * 64,
    'conformance_sha': 'e' * 64,
}
assert P_REPRO(GREEN_P) is True
P_REPRO(GREEN_P)


## C2.6 - Bundled predicate `PackReady_0_1_2`

`PackReady_0_1_2 = V_CORRECT AND A_SELFDESC AND D_CLOSED AND S_REAL AND P_REPRO`. Any false clause forces `NO_GO`.

In [ ]:
def PackReady_0_1_2(v, a, d, s, p):
    return V_CORRECT(v) and A_SELFDESC(a) and D_CLOSED(d) and S_REAL(s) and P_REPRO(p)

assert PackReady_0_1_2(GREEN_V, GREEN_A, GREEN_D, GREEN_S, GREEN_P) is True
PackReady_0_1_2(GREEN_V, GREEN_A, GREEN_D, GREEN_S, GREEN_P)


## C2.7 - Negative cases

Break each clause in isolation and prove `PackReady_0_1_2` collapses to `False`.

In [ ]:
def negate_v(v):
    bad = dict(v); bad['manifest_version'] = '1.1.1'; return bad
def negate_v_runtime(v):
    bad = dict(v); bad['runtime_version_constant'] = '0.1.0'; return bad
def negate_v_active_claim(v):
    bad = dict(v); bad['active_release_claims'] = {'1.1.1': True}; return bad
def negate_a_missing_manifest(a):
    bad = dict(a); bad['files_at_installed_root'] = [f for f in bad['files_at_installed_root'] if f != 'manifest.json']; return bad
def negate_a_source_repo(a):
    bad = dict(a); bad['reads_source_repo'] = True; return bad
def negate_a_repo_root_env(a):
    bad = dict(a); bad['reads_m365_repo_root_env'] = True; return bad
def negate_d_no_manifest(d):
    bad = dict(d); bad['declared_dependency_manifest_present'] = False; return bad
def negate_d_no_probe(d):
    bad = dict(d); bad['startup_probe_present'] = False; return bad
def negate_d_raw_mnf(d):
    bad = dict(d); bad['raw_module_not_found_observed'] = True; bad['missing_module_outcome_class'] = 'module_not_found_error'; return bad
def negate_d_repo_venv(d):
    bad = dict(d); bad['depends_on_repo_dot_venv'] = True; return bad
def negate_s_patched(s):
    bad = dict(s); bad['http_runtime_invoke_patched'] = True; return bad
def negate_s_no_subprocess(s):
    bad = dict(s); bad['launches_runtime_subprocess'] = False; return bad
def negate_s_no_alias(s):
    bad = dict(s); bad['legacy_alias_proven_over_http'] = False; return bad
def negate_s_mock_transport(s):
    bad = dict(s); bad['http_transport_class'] = 'mock_transport'; return bad
def negate_p_hashes(p):
    bad = dict(p); bad['build_b_bundle_sha'] = 'f' * 64; return bad
def negate_p_dirty_claim(p):
    bad = dict(p); bad['source_clean'] = False; bad['claims_clean'] = True; return bad
def negate_p_missing_key(p):
    bad = dict(p); bad.pop('dependency_lock_sha'); return bad

negative_cases = [
    ('V_manifest_version_wrong', negate_v(GREEN_V), GREEN_A, GREEN_D, GREEN_S, GREEN_P),
    ('V_runtime_constant_wrong', negate_v_runtime(GREEN_V), GREEN_A, GREEN_D, GREEN_S, GREEN_P),
    ('V_active_release_1_1_1', negate_v_active_claim(GREEN_V), GREEN_A, GREEN_D, GREEN_S, GREEN_P),
    ('A_missing_manifest_at_root', GREEN_V, negate_a_missing_manifest(GREEN_A), GREEN_D, GREEN_S, GREEN_P),
    ('A_reads_source_repo', GREEN_V, negate_a_source_repo(GREEN_A), GREEN_D, GREEN_S, GREEN_P),
    ('A_reads_m365_repo_root_env', GREEN_V, negate_a_repo_root_env(GREEN_A), GREEN_D, GREEN_S, GREEN_P),
    ('D_no_dependency_manifest', GREEN_V, GREEN_A, negate_d_no_manifest(GREEN_D), GREEN_S, GREEN_P),
    ('D_no_startup_probe', GREEN_V, GREEN_A, negate_d_no_probe(GREEN_D), GREEN_S, GREEN_P),
    ('D_raw_module_not_found', GREEN_V, GREEN_A, negate_d_raw_mnf(GREEN_D), GREEN_S, GREEN_P),
    ('D_depends_on_repo_dot_venv', GREEN_V, GREEN_A, negate_d_repo_venv(GREEN_D), GREEN_S, GREEN_P),
    ('S_http_runtime_invoke_patched', GREEN_V, GREEN_A, GREEN_D, negate_s_patched(GREEN_S), GREEN_P),
    ('S_no_subprocess', GREEN_V, GREEN_A, GREEN_D, negate_s_no_subprocess(GREEN_S), GREEN_P),
    ('S_no_legacy_alias_over_http', GREEN_V, GREEN_A, GREEN_D, negate_s_no_alias(GREEN_S), GREEN_P),
    ('S_mock_transport', GREEN_V, GREEN_A, GREEN_D, negate_s_mock_transport(GREEN_S), GREEN_P),
    ('P_hash_mismatch', GREEN_V, GREEN_A, GREEN_D, GREEN_S, negate_p_hashes(GREEN_P)),
    ('P_claims_clean_while_dirty', GREEN_V, GREEN_A, GREEN_D, GREEN_S, negate_p_dirty_claim(GREEN_P)),
    ('P_missing_required_key', GREEN_V, GREEN_A, GREEN_D, GREEN_S, negate_p_missing_key(GREEN_P)),
]

negative_results = []
for name, v, a, d, s, p in negative_cases:
    res = PackReady_0_1_2(v, a, d, s, p)
    assert res is False, f'Negative case {name} did not flip PackReady to False'
    negative_results.append({'case': name, 'pack_ready': res})

len(negative_results)


## C2.8 - Emit deterministic verification JSON

Emits `configs/generated/standalone_graph_runtime_pack_0_1_2_readiness_fix_v1_verification.json` recording the green base case, the per-invariant pass status, and every negative case result.

In [ ]:
verification = {
    'lemma_id': 'L99_m365_standalone_graph_runtime_pack_0_1_2_readiness_fix_v1',
    'plan_reference': 'plan:m365-standalone-graph-runtime-pack-0-1-2-readiness-fix:C2',
    'mode': 'deterministic-static',
    'seed_lock': {
        'pythonhashseed': PYTHONHASHSEED,
        'python_random_seed': PYTHON_RANDOM_SEED,
        'numpy_seed': NUMPY_SEED,
        'mode': 'deterministic-static',
        'seed': 0,
    },
    'target': TARGET,
    'green_base_case': {
        'V_CORRECT': V_CORRECT(GREEN_V),
        'A_SELFDESC': A_SELFDESC(GREEN_A),
        'D_CLOSED': D_CLOSED(GREEN_D),
        'S_REAL': S_REAL(GREEN_S),
        'P_REPRO': P_REPRO(GREEN_P),
        'PackReady_0_1_2': PackReady_0_1_2(GREEN_V, GREEN_A, GREEN_D, GREEN_S, GREEN_P),
    },
    'negative_cases': negative_results,
    'overall_ok': True,
}
out = Path('configs/generated/standalone_graph_runtime_pack_0_1_2_readiness_fix_v1_verification.json')
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(verification, indent=2, sort_keys=True), encoding='utf-8')
verification['overall_ok']
